# Week 1 — DINOv2 × DESI Galaxy Zoo 10

This notebook walks through extracting self-supervised visual features from galaxy cutouts  
using **DINOv2** (`facebook/dinov2-base`) on the **GZ10** dataset from the  
[Multimodal Universe](https://huggingface.co/datasets/MultimodalUniverse/gz10) collection.

The dataset provides **160 × 160 pixel RGB cutouts** of galaxies, along with  
morphology labels (`gz10_label`) and redshift values.  
The original imaging comes from four DESI optical filters (g, r, i, z),  
but this particular Hugging Face split ships a pre-composed 3-channel RGB thumbnail.

---
## 1. Environment Setup

In [ ]:
# Install required packages (uncomment if running for the first time)
# %pip install --quiet datasets transformers torch torchvision matplotlib scikit-learn tqdm Pillow

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image
from tqdm.auto import tqdm

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device    : {DEVICE}")

---
## 2. Load the DESI / Galaxy Zoo 10 Dataset

In [ ]:
from datasets import load_dataset

# The GZ10 dataset from the Multimodal Universe collection.
# Contains 160×160 pixel galaxy cutouts as RGB images.
dataset = load_dataset("MultimodalUniverse/gz10", split="train", trust_remote_code=True)

print(f"Dataset loaded successfully — {len(dataset)} samples")

---
## 3. Dataset Exploration

In [ ]:
# ---- Overview ----
print("=" * 60)
print("DATASET STRUCTURE")
print("=" * 60)
print(dataset)
print()

# ---- Features ----
print("FEATURES")
print("-" * 60)
for name, feat in dataset.features.items():
    print(f"  {name:30s}  →  {feat}")
print()

# ---- Column names ----
print(f"Column names : {dataset.column_names}")
print(f"Total samples: {len(dataset)}")

# ---- Peek at the first sample ----
sample_0 = dataset[0]
print(f"\nFirst sample keys: {list(sample_0.keys())}")

In [ ]:
# Inspect each field in the first sample to understand data types & shapes
sample = dataset[0]

for key in sample.keys():
    val = sample[key]
    if isinstance(val, np.ndarray):
        print(f"  '{key}' → ndarray  shape={val.shape}  dtype={val.dtype}")
    elif isinstance(val, Image.Image):
        print(f"  '{key}' → PIL.Image  size={val.size}  mode={val.mode}")
    elif isinstance(val, dict):
        print(f"  '{key}' → dict  sub-keys={list(val.keys())}")
    elif isinstance(val, (list, tuple)):
        print(f"  '{key}' → {type(val).__name__}  len={len(val)}")
    else:
        print(f"  '{key}' → {type(val).__name__}: {str(val)[:120]}")

---
## 4. Visualize the Galaxy Image Channels

The images are stored under the `rgb_image` key.  
They originate from four DESI optical filters (g, r, i, z) but are delivered  
as a pre-composed **3-channel RGB** thumbnail.  
We plot the R, G, B channels side-by-side and then the full colour composite.

In [ ]:
def get_image_array(sample):
    """
    Extract the galaxy image from a dataset sample as a numpy array.

    Handles multiple storage layouts:
      - 'rgb_image' key  (PIL Image or ndarray)  — GZ10 default
      - 'image' key      (PIL Image, ndarray, or nested dict)
      - Fallback: first ndarray with ndim >= 2

    Returns
    -------
    img : np.ndarray, shape (C, H, W), dtype float32
        C = 3 for RGB, C = 4 if the dataset stores raw g/r/i/z bands.
    """
    raw = None

    # --- Priority 1: 'rgb_image' (used by MultimodalUniverse/gz10) ---
    if "rgb_image" in sample:
        raw = sample["rgb_image"]
    # --- Priority 2: 'image' ---
    elif "image" in sample:
        raw = sample["image"]
    else:
        # Fallback: first PIL or ndarray field
        for key in sample:
            v = sample[key]
            if isinstance(v, Image.Image):
                raw = v
                break
            if isinstance(v, np.ndarray) and v.ndim >= 2:
                raw = v
                break

    if raw is None:
        raise ValueError(f"Cannot locate image. Keys: {list(sample.keys())}")

    # Convert PIL → numpy
    if isinstance(raw, Image.Image):
        raw = np.array(raw)

    # Handle nested dict (e.g. {'array': ..., 'path': ...})
    if isinstance(raw, dict):
        if "array" in raw:
            raw = np.array(raw["array"])
        elif "bytes" in raw:
            import io
            raw = np.array(Image.open(io.BytesIO(raw["bytes"])))
        else:
            raise ValueError(f"Unexpected dict layout: {list(raw.keys())}")

    img = np.array(raw, dtype=np.float32)

    # Normalise to (C, H, W)
    if img.ndim == 2:
        img = img[np.newaxis]                      # (1, H, W)
    elif img.ndim == 3:
        if img.shape[-1] in (1, 3, 4):             # (H, W, C) → (C, H, W)
            img = np.transpose(img, (2, 0, 1))
        # else assume already (C, H, W)

    return img


# ---- Extract the first galaxy ----
img = get_image_array(dataset[0])
n_channels = img.shape[0]
print(f"Image tensor shape: {img.shape}  ({n_channels} channels, {img.shape[1]}×{img.shape[2]} px)")
print(f"Value range       : [{img.min():.1f}, {img.max():.1f}]")

In [ ]:
# ---- Plot the individual channels side-by-side ----
# For 3-channel RGB images the channels map to R / G / B.
# For 4-channel images they would map to g / r / i / z.

if n_channels == 3:
    band_names = ["Red", "Green", "Blue"]
elif n_channels == 4:
    band_names = ["g-band", "r-band", "i-band", "z-band"]
else:
    band_names = [f"Ch {i}" for i in range(n_channels)]

fig, axes = plt.subplots(1, n_channels, figsize=(4 * n_channels, 4))
if n_channels == 1:
    axes = [axes]

for idx, ax in enumerate(axes):
    band = img[idx]
    vmin, vmax = np.percentile(band, [1, 99])
    ax.imshow(band, cmap="gray", origin="lower", vmin=vmin, vmax=vmax)
    ax.set_title(band_names[idx], fontsize=13)
    ax.axis("off")

fig.suptitle("Galaxy cutout — individual channels", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ---- Full colour composite ----
def to_display_rgb(img_chw):
    """
    Convert a (C, H, W) float image to a (H, W, 3) uint8 array for display.
    Works for both 3-channel RGB and 4-channel g/r/i/z.
    """
    if img_chw.shape[0] == 3:
        rgb = np.transpose(img_chw, (1, 2, 0))        # already RGB
    elif img_chw.shape[0] >= 4:
        # i / r / g → R / G / B  (astronomers' convention)
        rgb = np.stack([img_chw[2], img_chw[1], img_chw[0]], axis=-1)
    else:
        rgb = np.stack([img_chw[0]] * 3, axis=-1)      # grayscale → RGB

    # Per-channel percentile stretch → [0, 255]
    for c in range(3):
        lo, hi = np.percentile(rgb[..., c], [1, 99.5])
        rgb[..., c] = np.clip((rgb[..., c] - lo) / (hi - lo + 1e-8), 0, 1)

    return (rgb * 255).astype(np.uint8)


fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(to_display_rgb(img), origin="lower")
ax.set_title("RGB colour composite", fontsize=13)
ax.axis("off")
plt.show()

---
## 5. Load Pretrained DINOv2

In [ ]:
from transformers import AutoModel, AutoImageProcessor

MODEL_NAME = "facebook/dinov2-base"

# Image processor handles resizing, normalisation, and conversion to tensors
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)

# Load the base Vision Transformer backbone (no classification head)
model = AutoModel.from_pretrained(MODEL_NAME)
model = model.to(DEVICE)
model.eval()

print(f"Model        : {MODEL_NAME}")
print(f"Hidden dim   : {model.config.hidden_size}")
print(f"Patch size   : {model.config.patch_size}")
print(f"Num layers   : {model.config.num_hidden_layers}")
print(f"Parameters   : {sum(p.numel() for p in model.parameters()) / 1e6:.1f} M")

---
## 6. Preprocess Image (Handle Channel Bottleneck)

DINOv2 expects standard **3-channel RGB** inputs.  

- If the image is already 3-channel (as in this GZ10 split), we use it directly.  
- If it were 4-channel (raw g/r/i/z), we would drop the z-band or average  
  adjacent pairs to produce a 3-channel representation.  

The function below handles both cases, then applies a percentile stretch  
and passes the result through the `AutoImageProcessor`.

In [ ]:
def preprocess_galaxy(img_chw, processor, strategy="drop_z"):
    """
    Convert a (C, H, W) galaxy image to a DINOv2-ready tensor.

    Parameters
    ----------
    img_chw : np.ndarray, shape (C, H, W)
        Galaxy cutout.  C = 3 (RGB) or C = 4 (g/r/i/z).
    processor : AutoImageProcessor
        Hugging Face image processor for DINOv2.
    strategy : str  (only matters when C == 4)
        'drop_z'   — use g, r, i as the three channels.
        'drop_g'   — use r, i, z.
        'average'  — average adjacent pairs: (g+r)/2, (r+i)/2, (i+z)/2.

    Returns
    -------
    pixel_values : torch.Tensor, shape (1, 3, 224, 224)
    rgb_uint8    : np.ndarray, shape (H, W, 3), dtype uint8
    """
    C = img_chw.shape[0]

    if C == 3:
        # Already RGB — just transpose to (H, W, C)
        rgb = np.transpose(img_chw, (1, 2, 0)).copy()
    elif C >= 4:
        # 4-channel → 3-channel
        if strategy == "drop_z":
            rgb = np.stack([img_chw[0], img_chw[1], img_chw[2]], axis=-1)
        elif strategy == "drop_g":
            rgb = np.stack([img_chw[1], img_chw[2], img_chw[3]], axis=-1)
        elif strategy == "average":
            rgb = np.stack([
                (img_chw[0] + img_chw[1]) / 2.0,
                (img_chw[1] + img_chw[2]) / 2.0,
                (img_chw[2] + img_chw[3]) / 2.0,
            ], axis=-1)
        else:
            raise ValueError(f"Unknown strategy: {strategy}")
    else:
        # 1- or 2-channel: replicate to 3
        rgb = np.stack([img_chw[0]] * 3, axis=-1)

    # Percentile stretch → [0, 1] → uint8
    for c in range(3):
        lo, hi = np.percentile(rgb[..., c], [1, 99.5])
        rgb[..., c] = np.clip((rgb[..., c] - lo) / (hi - lo + 1e-8), 0, 1)

    rgb_uint8 = (rgb * 255).astype(np.uint8)

    # AutoImageProcessor expects a PIL Image or uint8 array (H, W, 3)
    inputs = processor(images=Image.fromarray(rgb_uint8), return_tensors="pt")
    pixel_values = inputs["pixel_values"].to(DEVICE)   # (1, 3, 224, 224)

    return pixel_values, rgb_uint8


# ---- Demo on the first image ----
img = get_image_array(dataset[0])
pixel_values, rgb_preview = preprocess_galaxy(img, processor)

print(f"Input channels         : {img.shape[0]}")
print(f"Processor output shape : {pixel_values.shape}")
print(f"Pixel value range      : [{pixel_values.min():.2f}, {pixel_values.max():.2f}]")

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(rgb_preview, origin="lower")
ax.set_title("3-channel preview (after preprocessing)", fontsize=12)
ax.axis("off")
plt.show()

---
## 7. Run Inference

In [ ]:
with torch.no_grad():
    outputs = model(pixel_values, output_hidden_states=True)

# The main output is `last_hidden_state`, shape (batch, num_tokens, hidden_dim)
last_hidden = outputs.last_hidden_state
print(f"last_hidden_state shape: {last_hidden.shape}")
print(f"  → batch size   : {last_hidden.shape[0]}")
print(f"  → total tokens : {last_hidden.shape[1]}  (1 CLS + N² patches)")
print(f"  → hidden dim   : {last_hidden.shape[2]}")

---
## 8. Extract CLS Token

The **CLS token** (index 0) is the global, image-level representation learned by the ViT.  
It aggregates information from all patches and is the standard choice for  
downstream classification or retrieval.

In [ ]:
cls_token = last_hidden[:, 0, :]           # shape: (1, 768)

print(f"CLS token shape : {cls_token.shape}")
print(f"CLS token stats : mean={cls_token.mean():.4f}, std={cls_token.std():.4f}")

---
## 9. Extract Patch Tokens

Everything after the CLS token corresponds to spatial **patch embeddings**.  
For `dinov2-base` with patch size 14 and input 224 × 224,  
we get 16 × 16 = **256 patch tokens**.

In [ ]:
patch_tokens = last_hidden[:, 1:, :]       # shape: (1, N_patches, 768)

print(f"Patch tokens shape : {patch_tokens.shape}")
print(f"Patch tokens stats : mean={patch_tokens.mean():.4f}, std={patch_tokens.std():.4f}")

---
## 10. Inspect Tensor Dimensions

In [ ]:
num_patches = patch_tokens.shape[1]
grid_side   = int(np.sqrt(num_patches))
hidden_dim  = patch_tokens.shape[2]

print("=" * 50)
print("TENSOR DIMENSION SUMMARY")
print("=" * 50)
print(f"CLS  token  : {cls_token.shape}         →  1 × {hidden_dim}")
print(f"Patch tokens: {patch_tokens.shape}  →  {num_patches} patches × {hidden_dim}")
print(f"Spatial grid : {grid_side} × {grid_side} = {grid_side**2} patches")
print(f"Hidden dim   : {hidden_dim}  {'✓' if hidden_dim == 768 else '✗ (expected 768)'}")

---
## 11. Visualize Patch Grid (PCA → RGB)

We reshape the flat sequence of patch embeddings back onto a 2D spatial grid,  
then reduce 768-d → 3-d with **PCA** and map the components to RGB.  
This reveals which spatial regions share similar learned features.

In [ ]:
from sklearn.decomposition import PCA

def visualize_patch_pca(patch_tokens_tensor, grid_side, title="PCA of patch tokens"):
    """
    Project patch tokens to 3-d with PCA and render as an RGB image
    on the spatial grid.

    Parameters
    ----------
    patch_tokens_tensor : torch.Tensor, shape (1, N, D)
    grid_side           : int – side length of the square patch grid
    """
    tokens_np = patch_tokens_tensor.squeeze(0).cpu().numpy()   # (N, D)

    # Fit PCA to reduce 768-d → 3-d
    pca = PCA(n_components=3)
    tokens_3d = pca.fit_transform(tokens_np)                   # (N, 3)

    # Min-max normalise each component to [0, 1] for RGB mapping
    for c in range(3):
        lo, hi = tokens_3d[:, c].min(), tokens_3d[:, c].max()
        tokens_3d[:, c] = (tokens_3d[:, c] - lo) / (hi - lo + 1e-8)

    # Reshape to spatial grid (grid_side, grid_side, 3)
    pca_grid = tokens_3d.reshape(grid_side, grid_side, 3)

    explained = pca.explained_variance_ratio_ * 100

    # ---- Plot ----
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))

    # Full RGB composite
    axes[0].imshow(pca_grid, interpolation="nearest")
    axes[0].set_title("PCA RGB composite", fontsize=12)
    axes[0].axis("off")

    # Individual PCA components
    for c_idx in range(3):
        im = axes[c_idx + 1].imshow(
            pca_grid[..., c_idx], cmap="viridis", interpolation="nearest"
        )
        axes[c_idx + 1].set_title(
            f"PC{c_idx + 1}  ({explained[c_idx]:.1f}% var)", fontsize=12
        )
        axes[c_idx + 1].axis("off")
        plt.colorbar(im, ax=axes[c_idx + 1], fraction=0.046, pad=0.04)

    fig.suptitle(title, fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

    return pca, pca_grid


# ---- Run on the first galaxy ----
pca_model, pca_grid = visualize_patch_pca(
    patch_tokens, grid_side,
    title="DINOv2 patch features — Galaxy #0"
)

In [ ]:
# Side-by-side: original image vs. DINOv2 feature map
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

ax1.imshow(rgb_preview, origin="lower")
ax1.set_title("Input image", fontsize=13)
ax1.axis("off")

ax2.imshow(pca_grid, interpolation="nearest")
ax2.set_title(f"DINOv2 patch PCA  ({grid_side}×{grid_side})", fontsize=13)
ax2.axis("off")

plt.suptitle("What is DINOv2 paying attention to?", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Visualise a few more galaxies for comparison
N_EXAMPLES = 4
fig, axes = plt.subplots(2, N_EXAMPLES, figsize=(4 * N_EXAMPLES, 8))

for col_idx in range(N_EXAMPLES):
    idx = col_idx * 10   # spread across the dataset
    img_i = get_image_array(dataset[idx])
    pv_i, rgb_i = preprocess_galaxy(img_i, processor)

    with torch.no_grad():
        out_i = model(pv_i)
    ptk_i = out_i.last_hidden_state[:, 1:, :]

    tok_np = ptk_i.squeeze(0).cpu().numpy()
    pca_3 = PCA(n_components=3).fit_transform(tok_np)
    for c in range(3):
        lo, hi = pca_3[:, c].min(), pca_3[:, c].max()
        pca_3[:, c] = (pca_3[:, c] - lo) / (hi - lo + 1e-8)
    grid_img = pca_3.reshape(grid_side, grid_side, 3)

    axes[0, col_idx].imshow(rgb_i, origin="lower")
    axes[0, col_idx].set_title(f"Galaxy #{idx}", fontsize=11)
    axes[0, col_idx].axis("off")

    axes[1, col_idx].imshow(grid_img, interpolation="nearest")
    axes[1, col_idx].set_title("DINOv2 PCA", fontsize=11)
    axes[1, col_idx].axis("off")

axes[0, 0].set_ylabel("Input", fontsize=13)
axes[1, 0].set_ylabel("Features", fontsize=13)
plt.suptitle("DINOv2 feature maps across galaxies", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

---
## 12. Initial Embedding Exploration (100 galaxies → t-SNE / PCA)

We extract CLS embeddings for the first 100 galaxies and project them  
into 2-D with both **PCA** and **t-SNE** to see how DINOv2 naturally  
clusters different morphologies.

In [ ]:
N_SAMPLES = 100

cls_embeddings = []   # will collect (768,) vectors
rgb_thumbs     = []   # small thumbnails for the scatter plot

for i in tqdm(range(N_SAMPLES), desc="Extracting CLS embeddings"):
    img_i = get_image_array(dataset[i])
    pv_i, rgb_i = preprocess_galaxy(img_i, processor)

    with torch.no_grad():
        out_i = model(pv_i)

    cls_i = out_i.last_hidden_state[:, 0, :].squeeze(0).cpu().numpy()  # (768,)
    cls_embeddings.append(cls_i)
    rgb_thumbs.append(rgb_i)

cls_matrix = np.stack(cls_embeddings)   # (100, 768)
print(f"\nCLS embedding matrix: {cls_matrix.shape}")

In [ ]:
from sklearn.manifold import TSNE

# ---- PCA 2-D ----
pca_2d = PCA(n_components=2).fit_transform(cls_matrix)

# ---- t-SNE 2-D ----
tsne_2d = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate="auto",
    init="pca",
    random_state=42,
).fit_transform(cls_matrix)

print(f"PCA  projection shape : {pca_2d.shape}")
print(f"t-SNE projection shape: {tsne_2d.shape}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# ---- PCA scatter ----
scatter1 = ax1.scatter(
    pca_2d[:, 0], pca_2d[:, 1],
    c=np.arange(N_SAMPLES), cmap="Spectral", s=40, alpha=0.8, edgecolors="k", linewidth=0.3
)
ax1.set_xlabel("PC 1", fontsize=12)
ax1.set_ylabel("PC 2", fontsize=12)
ax1.set_title("PCA of CLS Embeddings", fontsize=14)
plt.colorbar(scatter1, ax=ax1, label="Sample index")

# ---- t-SNE scatter ----
scatter2 = ax2.scatter(
    tsne_2d[:, 0], tsne_2d[:, 1],
    c=np.arange(N_SAMPLES), cmap="Spectral", s=40, alpha=0.8, edgecolors="k", linewidth=0.3
)
ax2.set_xlabel("t-SNE 1", fontsize=12)
ax2.set_ylabel("t-SNE 2", fontsize=12)
ax2.set_title("t-SNE of CLS Embeddings", fontsize=14)
plt.colorbar(scatter2, ax=ax2, label="Sample index")

fig.suptitle(
    f"DINOv2 CLS Embeddings — {N_SAMPLES} galaxies (no fine-tuning)",
    fontsize=15, y=1.02
)
plt.tight_layout()
plt.show()

In [ ]:
# ---- t-SNE with galaxy thumbnails overlaid ----
from matplotlib.offsetbox import OffsetImage, AnnotationBbox

fig, ax = plt.subplots(figsize=(14, 14))
ax.scatter(tsne_2d[:, 0], tsne_2d[:, 1], alpha=0.0)  # invisible anchors

for i in range(N_SAMPLES):
    thumb = rgb_thumbs[i][::4, ::4, :]  # downsample for speed
    im = OffsetImage(thumb, zoom=1.0)
    ab = AnnotationBbox(im, (tsne_2d[i, 0], tsne_2d[i, 1]),
                        frameon=False, pad=0)
    ax.add_artist(ab)

ax.set_xlabel("t-SNE 1", fontsize=13)
ax.set_ylabel("t-SNE 2", fontsize=13)
ax.set_title(
    f"t-SNE of DINOv2 CLS embeddings — {N_SAMPLES} galaxies (thumbnail overlay)",
    fontsize=15
)
plt.tight_layout()
plt.show()

---
### Key Take-aways

| Observation | Detail |
|---|---|
| **CLS embedding** | A single 768-d vector per galaxy that captures global morphology. |
| **Patch features** | 256 spatially-arranged 768-d vectors revealing local structure. |
| **Zero-shot clustering** | Even without fine-tuning, DINOv2 groups visually similar galaxies together in the t-SNE plot. |
| **Next steps** | Fine-tune a linear probe on top of the CLS token for galaxy morphology classification, or use the patch tokens for segmentation. |